In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from matplotlib import rcParams
plt.rcParams['font.family'] = 'SimHei' 
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
df001 = pd.read_csv('data.csv')
missing_count = df001.isna().sum().tolist()
column = df001.columns.tolist()
df002 = pd.DataFrame({'column':column, 'missing_count':missing_count})
df002.to_csv('001missing_summary.csv', index=False)
df003 = df001.dropna()
df003['co2'] = df003['co2'] / 10
df003.to_csv('002cleaned_data.csv', index=False)
df004 = pd.read_csv('002cleaned_data.csv')
df005 = df004.groupby(['country', 'year']).agg(
    co2=('co2', 'mean'),
    co2_per_capita=('co2_per_capita', 'mean'),
    renewable_share=('renewable_share', 'mean'),
    population=('population', 'mean'),
    gdp=('gdp', 'mean'),
    region=('region', 'first')
).reset_index()
df005.to_csv('003merged_data.csv', index=False)
df006 = pd.read_csv('003merged_data.csv')
df006['renewable_growth_rate'] = df006.groupby('country')['renewable_share'].pct_change()
df006['emissions_intensity'] = df006['co2'] / df006['gdp']
df006.to_csv('003merged_data.csv', index=False)
def zscore_df(x):
    return (x - x.mean()) / x.std()
co2_per_capita_zscore = zscore_df(df006['co2_per_capita']).tolist()
renewable_growth_rate_zscore = zscore_df(df006['renewable_growth_rate']).tolist()
emissions_intensity_zscore = zscore_df(df006['emissions_intensity']).tolist()
df007 = pd.DataFrame({'co2_per_capita_zscore':co2_per_capita_zscore, 'renewable_growth_rate_zscore':renewable_growth_rate_zscore, 'emissions_intensity_zscore':emissions_intensity_zscore})
df007.to_csv('004processed_data.csv', index=False)

In [ ]:
df008 = pd.read_csv('003merged_data.csv')
df009 = df008.groupby(['country', 'year']).agg(
    mean_co2=('co2', 'mean'),
    std_co2=('co2', 'std'),
    min_co2=('co2', 'min'),
    max_co2=('co2', 'max'),
    mean_co2_per_capita=('co2_per_capita', 'mean'),
    std_co2_per_capita=('co2_per_capita', 'std'),
    min_co2_per_capita=('co2_per_capita', 'min'),
    max_co2_per_capita=('co2_per_capita', 'max'),
    mean_renewable_share=('renewable_share', 'mean'),
    std_renewable_share=('renewable_share', 'std'),
    min_renewable_share=('renewable_share', 'min'),
    max_renewable_share=('renewable_share', 'max')
).reset_index()
df009.to_csv('005descriptive_stats.csv', index=False)
df010 = pd.read_csv('003merged_data.csv')
def xielv(group):
    x = group['year']
    y = group['co2']
    slope, jieju = np.polyfit(x, y, 1)
    return slope
df011 = df010.groupby('country').apply(xielv).reset_index()
df011.columns = ['country', 'slope']
df011.to_csv('006country_trend.csv', index=False)
df012 = pd.read_csv('003merged_data.csv')
# 2. 计算co2年增长率（题目要求）
df012 = df012.sort_values(['country', 'year']).reset_index(drop=True)
df012['co2_growth_rate'] = df012.groupby('country')['co2'].pct_change()
def piersen(group):
    x = group['co2_growth_rate']
    y = group['renewable_share']
    mask = ~(x.isna() | y.isna())
    x_clean = x[mask]
    y_clean = y[mask]
    r, p = pearsonr(x_clean, y_clean)
    return r
df013 = df012.groupby('country').apply(piersen).reset_index()
df013.columns = ['country', 'correlation']
df013.to_csv('007correlation.csv', index=False)
df014 = pd.read_csv('003merged_data.csv')
def junfanggen(group):
    actual = group['co2']
    x = group['year']
    slope, jieju = np.polyfit(x, actual, 1)
    forecast = slope * x + jieju
    rmse = np.sqrt(np.mean((actual - forecast) ** 2))
    return pd.DataFrame({'country':group.name, 'year':x, 'actual':actual, 'forecast':forecast, 'rmse':rmse})
df015 = df014.groupby('country').apply(junfanggen).reset_index(drop=True)
df015.to_csv('008forecast_results.csv', index=False)

In [ ]:
data100 = pd.read_csv('003merged_data.csv')
data110 = data100.groupby('year').agg(
    total_co2=('co2', 'sum'),
    avg_renewable_share=('renewable_share', 'mean')
).reset_index()
data110.to_csv('009global_trends.csv')
fig, ax = plt.subplots(figsize=(12,6))
ax.plot(data110['year'], data110['total_co2'],  color='blue', marker='o', label='Total CO2 Emissions')
ax1 = ax.twinx()
ax1.plot(data110['year'], data110['avg_renewable_share'],   color='orange', marker='s', label='Average Renewable Share')
ax.set_xlabel('Year')
ax.set_ylabel('Total CO2 Emissions')
ax1.set_ylabel('Average Renewable Share')
plt.title('Total CO2 Emissions and Average Renewable Share Over Time')
# ✅ 关键：把两个轴的图例合并显示
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax1.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='best')
plt.tight_layout()
plt.savefig('global_trends.png')
plt.show()

In [ ]:
data111 = data100[data100['year'] == 2024][['country', 'year', 'co2', 'co2_per_capita','renewable_share','population', 'gdp', 'region']]
data111['bubble_size'] = data100['co2_per_capita'] * 10
data111.to_csv('010map_data.csv', index=False)
fig, ax4 = plt.subplots(figsize=(10,6))
scatter = ax4.scatter(data111['gdp'], data111['co2'], c=data111['renewable_share'], s=data111['bubble_size'], alpha=0.7, edgecolors='black')
# ---------------------- ✅ 关键：右侧添加颜色条 ----------------------
cbar = plt.colorbar(scatter, ax=ax4)  # 绑定颜色条到当前图
cbar.set_label('Average Renewable Share', fontsize=12)  # 颜色条标签
plt.title('Title')
plt.tight_layout()
plt.savefig('map_2024.png')
plt.show()


In [ ]:
data112 = data100[data100['country'].isin(['Country_001', 'Country_002', 'Country_003'])][['country', 'year', 'co2', 'co2_per_capita', 'renewable_share', 'population', 'gdp', 'region']]
data112.to_csv('011combo_data.csv', index=False)
data200 = data112[data112['country'] == 'Country_001'][['year', 'co2', 'renewable_share']]
data210 = data112[data112['country'] == 'Country_002'][['year', 'co2', 'renewable_share']]
data211 = data112[data112['country'] == 'Country_003'][['year', 'co2', 'renewable_share']]
fig, ax2 = plt.subplots(figsize=(10,5))
ax2.plot(data200['year'], data200['co2'], color='orange', label='Country_001 CO2', marker='o')
ax2.plot(data210['year'], data210['co2'], color='green', label='Country_002 CO2', marker='o')
ax2.plot(data211['year'], data211['co2'], color='blue', label='Country_003 CO2', marker='o')
ax3 = ax2.twinx()
ax3.plot(data200['year'], data200['renewable_share'], color='orange', linestyle='--', label='Country_001 Renewable Share', marker='s')
ax3.plot(data210['year'], data210['renewable_share'], color='green', linestyle='--', label='Country_002 Renewable Share', marker='s')
ax3.plot(data211['year'], data211['renewable_share'], color='blue', linestyle='--', label='Country_003 Renewable Share', marker='s')
ax2.set_xlabel('Year')
ax2.set_ylabel('CO2 Emissions')
ax3.set_ylabel('Renewable Share')
plt.title('CO2 Emissions and Renewable Share Over Time for Country_001, Country_002, Country_003')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax3.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='best')
plt.tight_layout()
plt.savefig('combo_chart.png')
plt.show()

In [ ]:
df100 = pd.read_csv('003merged_data.csv')
df110 = df100[df100['year'] == 2000][['country', 'co2', 'renewable_share']]
df111 = df100[df100['year'] == 2024][['country', 'co2', 'renewable_share']]
df111.columns = ['country', 'co2_2024', 'renewable_share_2024']
df110.columns = ['country', 'co2_2000', 'renewable_share_2000']
merged = pd.merge(df110, df111, on='country')
merged['change'] = merged['co2_2024'] - merged['co2_2000']
merged['avg_renewable_increase'] = merged['renewable_share_2024'] - merged['renewable_share_2000']
xiao = merged['change'].nsmallest().index.tolist()
df120 = merged.loc[xiao][['country', 'change', 'avg_renewable_increase']]
df120.to_csv('012top_reduction.csv', index=False)
df121 = pd.read_csv('004processed_data.csv')
df122 = df121.dropna()
kmeans = KMeans(
n_clusters=4, # 题目要求
init="k-means++", # 默认最佳
n_init=10, # 重复次数，避免局部最优
max_iter=300, # 最大迭代次数
tol=1e-4, # 收敛阈值
random_state=42 # 固定随机种子，保证复现
)
labels = pd.DataFrame(kmeans.fit_predict(df122), columns=['labels'])
centers = pd.DataFrame(kmeans.cluster_centers_, columns=df122.columns)
df200 = df121.loc[df122.index]
df100['labels'] = labels
df210 = df100.groupby('labels').agg(
    avg_co2=('co2', 'mean'),
    avg_renewable_share=('renewable_share', 'mean')
).reset_index()
df210.head()